# Credit Card Fraud Dataset Analysis - Comparing Raw data performance with Statistically preprocessed data performance

## Step 1 — Import Libraries

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             precision_recall_curve)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
warnings.filterwarnings("ignore")

In [3]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

# print("Path to dataset files:", path)


In [4]:
# df = pd.read_csv(f"{path}/creditcard.csv")

## Step 2 — Load Dataset

In [20]:
df = pd.read_csv("creditcard.csv")

print(df.shape)
print(df["Class"].value_counts())

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


## Step 3 — Train–Test Split (Stratified)

In [21]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## Step 4 — Define Evaluation Function

In [22]:
def evaluate_model(model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:,1]

    # threshold optimization
    precision, recall, thresholds = precision_recall_curve(y_test, probs)
    f1 = (2 * precision * recall) / (precision + recall + 1e-6)

    best_threshold = thresholds[np.argmax(f1)]

    preds = (probs >= best_threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
        "ROC_AUC": roc_auc_score(y_test, probs),
        "PR_AUC": average_precision_score(y_test, probs)
    }

    return results

## Step 5 — Define Raw Models

In [23]:
models_raw = {

    "Logistic": LogisticRegression(max_iter=200),

    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(n_neighbors=5),

    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=42
    )
}

## Step 6 — Train Raw Models

In [24]:
results_raw = {}

for name, model in models_raw.items():

    res = evaluate_model(model, X_train, y_train, X_test, y_test)

    results_raw[name] = res

raw_df = pd.DataFrame(results_raw).T

print("RAW RESULTS")
print(raw_df)

RAW RESULTS
              Accuracy  Precision    Recall        F1   ROC_AUC    PR_AUC
Logistic      0.999228   0.787234  0.755102  0.770833  0.943095  0.675841
RandomForest  0.999649   0.933333  0.857143  0.893617  0.962348  0.873052
KNN           0.998420   0.833333  0.102041  0.181818  0.636393  0.114919
XGBoost       0.999508   0.926829  0.775510  0.844444  0.938952  0.797291


## 7. Statistical Preprocessing Pipeline
### 7.1 Feature Scaling

Necessary for logistic regression and KNN.

In [25]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

## Step 7.2 Remove Multicollinearity Using VIF

In [26]:
def remove_high_vif(X, threshold=10):

    X = X.copy()

    while True:

        vif = pd.Series(
            [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
            index=X.columns
        )

        max_vif = vif.max()

        if max_vif > threshold:

            drop_feature = vif.idxmax()
            X = X.drop(columns=[drop_feature])

        else:
            break

    return X

X_train_vif = remove_high_vif(X_train_scaled)

X_test_vif = X_test_scaled[X_train_vif.columns]

## Step 7.3 SMOTE Oversampling

In [27]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_vif, y_train)

print("After SMOTE:", np.bincount(y_train_smote))

After SMOTE: [227451 227451]


## Step 8. Define Statistically Enhanced Models

In [28]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

models_stat = {

    "Logistic": LogisticRegression(
        class_weight="balanced",
        max_iter=200
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ),

    "KNN": KNeighborsClassifier(n_neighbors=7),

    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    )
}

## Step 9 — Train Statistically Enhanced Models

In [29]:
results_stat = {}

for name, model in models_stat.items():

    res = evaluate_model(
        model,
        X_train_smote,
        y_train_smote,
        X_test_vif,
        y_test
    )

    results_stat[name] = res

stat_df = pd.DataFrame(results_stat).T

print("STATISTICAL RESULTS")
print(stat_df)

STATISTICAL RESULTS
              Accuracy  Precision    Recall        F1   ROC_AUC    PR_AUC
Logistic      0.999421   0.849462  0.806122  0.827225  0.970466  0.720335
RandomForest  0.999526   0.890110  0.826531  0.857143  0.980517  0.856753
KNN           0.999017   0.669355  0.846939  0.747748  0.948379  0.589291
XGBoost       0.999508   0.906977  0.795918  0.847826  0.981829  0.826812


## Step 10 — Compare Results

In [30]:
comparison = pd.concat(
    [raw_df, stat_df],
    axis=1,
    keys=["Raw", "Statistical"]
)

print(comparison)

                   Raw                                                    \
              Accuracy Precision    Recall        F1   ROC_AUC    PR_AUC   
Logistic      0.999228  0.787234  0.755102  0.770833  0.943095  0.675841   
RandomForest  0.999649  0.933333  0.857143  0.893617  0.962348  0.873052   
KNN           0.998420  0.833333  0.102041  0.181818  0.636393  0.114919   
XGBoost       0.999508  0.926829  0.775510  0.844444  0.938952  0.797291   

             Statistical                                                    
                Accuracy Precision    Recall        F1   ROC_AUC    PR_AUC  
Logistic        0.999421  0.849462  0.806122  0.827225  0.970466  0.720335  
RandomForest    0.999526  0.890110  0.826531  0.857143  0.980517  0.856753  
KNN             0.999017  0.669355  0.846939  0.747748  0.948379  0.589291  
XGBoost         0.999508  0.906977  0.795918  0.847826  0.981829  0.826812  


## Step 11 — Cross-Validated Evaluation

In [31]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []

for train_idx, test_idx in skf.split(X, y):

    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()

    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    smote = SMOTE(random_state=42)

    X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

    model = LogisticRegression(
        class_weight="balanced",
        max_iter=200
    )

    model.fit(X_tr, y_tr)

    probs = model.predict_proba(X_te)[:,1]

    score = average_precision_score(y_te, probs)

    cv_scores.append(score)

print("Mean PR-AUC:", np.mean(cv_scores))

Mean PR-AUC: 0.728783324954375
